## Colab setup (run this first)

This notebook expects the `pqr/` package from this repo to be present. In **Google Colab**, clone the repo and `cd` into it before running the imports below.

In [ ]:
# --- Colab-only setup ---
# 1) Put your repo URL here
REPO_URL = "https://github.com/OWNER/REPO.git"  # <-- change this
REPO_DIR = "repo"  # local folder name

import os
import sys

# Clone if needed, then cd into repo
if not os.path.exists(REPO_DIR):
    !git clone "$REPO_URL" "$REPO_DIR"
os.chdir(REPO_DIR)

# Make this repo importable (so `import pqr` works)
sys.path.insert(0, os.path.abspath("."))

# 2) Install any extra packages (core `pqr` uses only stdlib; keep this for extensions)
!python -m pip install -q --upgrade pip

print("cwd:", os.getcwd())
print("pqr exists:", os.path.exists("pqr"))

## Private Query Refinement — Notebook Demo

This notebook demonstrates the reference implementation in `pqr/` for the core pseudocode in `Private_Query_Refinement.pdf`:

- **Algorithm 1**: *Private-Queries-Diff-Top-K-Explanation*
- **Algorithm 2**: *Find-Top-k-Explanations*

It runs on the included toy dataset (`pqr/toy.csv`) and prints the resulting **differentially private** (noisy) histograms explaining the differences between two queries.

In [1]:
import json
import random

from pqr.algorithm import generate_simple_predicate_views, private_queries_diff_topk_explanation
from pqr.query import Condition, Op, Query

Error: 

In [ ]:
# Load the toy dataset shipped with the repo
import csv
from pathlib import Path

csv_path = Path("pqr/toy.csv")
rows = []
with csv_path.open(newline="", encoding="utf-8") as f:
    for r in csv.DictReader(f):
        rows.append(dict(r))

len(rows), rows[0]

In [ ]:
# Define two queries Q1 and Q2 (selection predicates only)
# Q1: department == 'sales'
# Q2: department == 'engineering'
q1 = Query((Condition("department", Op.EQ, "sales"),))
q2 = Query((Condition("department", Op.EQ, "engineering"),))

# Attributes we want explanations for
attrs = ["gender", "seniority"]

# Predicate-space attributes used to generate view predicates
predicate_views = generate_simple_predicate_views(rows, predicate_attrs=["country"])

In [ ]:
rng = random.Random(7)

explanations = private_queries_diff_topk_explanation(
    dataset=rows,
    q1=q1,
    q2=q2,
    attributes=attrs,
    predicate_views=predicate_views,
    tau=3,
    k=2,
    epsilon=1.0,
    rng=rng,
)

print(json.dumps(explanations, indent=2, sort_keys=True))